In [ ]:
import bilby
import lal
import numpy as np
from matplotlib import pyplot as plt
import taichi as ti

from pespace.detector.antenna import InterferometerAntenna, FDResponseModelMarset2018
from pespace.detector.tdi import TDIChannelData, FDMichelsonConstantEqualArm
from pespace.detector.orbit import available_orbit_models
from pespace.inference.interface_bilby import LikelihoodBilbyInterface
from tiwave.waveforms import IMRPhenomXAS

ti.init(arch=ti.cpu, default_fp=ti.f64, cpu_max_num_threads=1, offline_cache=False)


In [ ]:
tdi_gen = "2.0"
tdi_chan = ("A", "E", "T")

dt = 10
f_min = 1e-4
f_max = 0.5*(1/dt)
f_ref = f_min
t_start = 0.0

num_tsamples = 2**np.ceil(np.log2(7*lal.DAYJUL_SI/dt))
duration = num_tsamples * dt
before_tc = 0.8 * duration
after_tc = 0.2 * duration
tc = t_start + before_tc
# tc = 0.0

params = dict(
    total_mass=3e6,
    mass_ratio=0.6,
    chi_1=0.75,
    chi_2=0.62,
    luminosity_distance=56000.0,
    inclination=0.4,
    reference_phase=1.3,
    ecliptic_longitude=1.375,
    ecliptic_latitude=-1.2108,
    polarization=2.659,
    coalescence_time=tc,
)
params = bilby.gw.conversion.generate_mass_parameters(params)
params


In [ ]:
tdi_data = TDIChannelData()
tdi_data.set_fd_data_from_zero(
    channels=tdi_chan, 
    duration=duration, 
    delta_time=dt,
    start_time=t_start,
    minimum_frequency=f_min,
    maximum_frequency=f_max,
    )
tdi_data.set_fd_noise_power_density_from_model("LISA_SciRDv1", tdi_generation=tdi_gen)
noise = tdi_data.get_fd_noise_realization()
tdi_data.add_into_fd_data(noise)

wf_xas = IMRPhenomXAS(
    tdi_data.frequency_samples, 
    f_ref, 
    parameter_conversion=bilby.gw.conversion.generate_component_masses
    )
wf_xas.update_waveform(params)

orbit_model = available_orbit_models['LISA_analytic']
response_model = FDResponseModelMarset2018()
tdi_combination = FDMichelsonConstantEqualArm(generation=tdi_gen, orthogonal=True)
lisa = InterferometerAntenna(
    name="lisa",
    tdi_data=tdi_data,
    orbit_model=orbit_model,
    response_model=response_model,
    tdi_combination=tdi_combination,
)
lisa.inject_signal(
    wf_xas.waveform_container,
    params["ecliptic_longitude"],
    params["ecliptic_latitude"],
    params["polarization"],
    params["coalescence_time"],
)

likelihood = LikelihoodBilbyInterface(
    waveform=wf_xas,
    detector=lisa,
    channels=('A', 'E', 'T')
    # channels=('A',)
    )


In [ ]:
display(params)
display(likelihood.parameters)

In [ ]:
params.pop('total_mass', None)
params.pop('mass_1', None)
params.pop('mass_2', None)
params.pop('symmetric_mass_ratio', None)
params

In [ ]:
likelihood.parameters.update(params)
ll = likelihood.log_likelihood()
ll

In [ ]:
psd_A = tdi_data.fd_noise_power_density_numpy['A']
signal_A = lisa.tdi_response_numpy["A"]
data_A = lisa.tdi_data.fd_data_numpy['A']

ll_np = -2 / duration * np.vdot((data_A-signal_A), (data_A-signal_A)/psd_A)
display(ll_np)
hh = -2 / duration * np.vdot(signal_A, signal_A/psd_A)
display(hh)
dd = -2 / duration * np.vdot(data_A, data_A/psd_A)
display(dd)
dh = 4 / duration * np.vdot(data_A, signal_A/psd_A).real
display(dh)
display(ll_np - (dd + hh + dh))


plt.figure()
plt.loglog(tdi_data.data_info.frequency_samples_array, np.abs(signal_A))
plt.loglog(tdi_data.data_info.frequency_samples_array, np.abs(data_A))


In [ ]:
ll_np = 0.0
for chan in likelihood.channels:
    psd = tdi_data.fd_noise_power_density_numpy[chan]
    signal = lisa.tdi_response_numpy[chan]
    data = lisa.tdi_data.fd_data_numpy[chan]

    ll_chan = -2 / duration * np.vdot((data-signal), (data-signal)/psd)
    ll_np += ll_chan

display(ll_np)
display(ll-ll_np.real)


In [ ]:
num = 1000
params_samples = {
    'chi_1': np.linspace(-0.99, 0.99, num),
    'chi_2': np.linspace(-0.99, 0.99, num),
    'ecliptic_longitude': np.linspace(0.0, 2*np.pi, num),
    'ecliptic_latitude': np.linspace(-np.pi/2, np.pi/2, num),
    'inclination': np.linspace(0.0, np.pi, num),
    'polarization': np.linspace(0.0, np.pi, num),
    'reference_phase': np.linspace(0.0, 2*np.pi, num),
    'coalescence_time': np.linspace(tc-1000, tc+1000, num),
    'chirp_mass': np.linspace(5e5, 2e6, num),
    'mass_ratio': np.linspace(0.05, 0.99, num),
    'luminosity_distance': np.exp(np.linspace(np.log(1e4), np.log(1e6), num)),
}

In [ ]:
for key, samples in params_samples.items():
    ll_samples = np.zeros(num)
    input_params = params.copy()

    for i in range(num):
        input_params.update({key: samples[i]})
        likelihood.parameters.update(input_params)
        ll_samples[i] = likelihood.log_likelihood()
    
    plt.figure()
    plt.title(key)
    plt.plot(samples, ll_samples)
    plt.axvline(params[key], linestyle='dashed', color='tab:red')
